# Notebook 1: Preprocessing — Feature Extraction

**Wildfire Susceptibility Mapping — Muğla Province, Turkey**  
CME434, Karabük University

**Purpose:** Load samplepoints.shp + 15 feature rasters, extract values at each point, save `Inputs.txt` + `Label.txt`  
**Input:** `GIS_Wildfire_Mugla/samplepoints.shp`, `GIS_Wildfire_Mugla/01_elevation.tif` … `15_rainfall.tif`  
**Output:** `Inputs.txt` (1000×15), `Label.txt` (1000,)

## Step 1 — Mount Google Drive

In [21]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/GIS_Wildfire_Mugla'
print('Drive mounted. Folder:', DRIVE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Folder: /content/drive/MyDrive/GIS_Wildfire_Mugla


## Step 2 — Install / Import Libraries

In [22]:
!pip install rasterio geopandas --quiet

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.sample import sample_gen
import os

print(f'rasterio {rasterio.__version__}  |  geopandas {gpd.__version__}')

rasterio 1.5.0  |  geopandas 1.1.3


## Step 3 — Load Sample Points

In [24]:
shp_path = f'{DRIVE}/samplepoints.shp'
gdf = gpd.read_file(shp_path)
print(f'Loaded {len(gdf)} sample points')
print(f'CRS: {gdf.crs}')
print(gdf[['Label','geometry']].head())
print('\nClass balance:')
print(gdf['Label'].value_counts().rename({1:'Burned', 0:'Unburned'}))

Loaded 1045 sample points
CRS: EPSG:4326
   Label                   geometry
0      1  POINT (27.38752 37.54193)
1      1  POINT (27.38216 37.54146)
2      1  POINT (27.38687 37.54634)
3      1  POINT (27.38156 37.54586)
4      1  POINT (27.38212 37.11839)

Class balance:
Label
Unburned    616
Burned      429
Name: count, dtype: int64


## Step 4 — Define Feature Rasters

In [26]:
FEATURES = [
    ('elevation',     '01_elevation.tif'),
    ('slope',         '02_slope.tif'),
    ('aspect',        '03_aspect.tif'),
    ('hillshade',     '04_hillshade.tif'),
    ('tpi',           '05_tpi.tif'),
    ('ndvi',          '06_ndvi.tif'),
    ('ndwi',          '07_ndwi.tif'),
    ('evi',           '08_evi.tif'),
    ('nbr',           '09_nbr.tif'),
    ('wind_speed',    '10_wind_speed.tif'),
    ('lst',           '11_lst.tif'),
    ('dist_roads',    '13_dist_roads.tif'),
    ('dist_urban',    '14_dist_urban.tif'),
    ('rainfall',      '15_rainfall.tif'),
    ('soil_moisture', '12_soil_moisture.tif'),  # last — optional, skip if missing
]

for name, fname in FEATURES:
    path = f'{DRIVE}/{fname}'
    exists = '✓' if os.path.exists(path) else '✗ MISSING'
    print(f'  {exists}  {fname}')

  ✓  01_elevation.tif
  ✓  02_slope.tif
  ✓  03_aspect.tif
  ✓  04_hillshade.tif
  ✓  05_tpi.tif
  ✓  06_ndvi.tif
  ✓  07_ndwi.tif
  ✓  08_evi.tif
  ✓  09_nbr.tif
  ✓  10_wind_speed.tif
  ✓  11_lst.tif
  ✓  13_dist_roads.tif
  ✓  14_dist_urban.tif
  ✓  15_rainfall.tif
  ✓  12_soil_moisture.tif


## Step 5 — Extract Raster Values at Sample Points

In [32]:
def extract_values(gdf, raster_path, col_name):
    with rasterio.open(raster_path) as src:
        pts = gdf.to_crs(src.crs) if gdf.crs != src.crs else gdf
        coords = [(geom.x, geom.y) for geom in pts.geometry]
        values = [v[0] for v in src.sample(coords)]
    return values

df = gdf[['Label']].copy().reset_index(drop=True)
extracted = []
skipped   = []

for col_name, fname in FEATURES:
    path = f'{DRIVE}/{fname}'
    if not os.path.exists(path):
        print(f'  ⚠ SKIPPED {fname} — file not found')
        skipped.append(col_name)
        continue
    try:
        vals = extract_values(gdf, path, col_name)
        df[col_name] = vals
        extracted.append(col_name)
        print(f'  ✓ {col_name}: min={min(vals):.3f}  max={max(vals):.3f}')
    except Exception as e:
        print(f'  ✗ ERROR {fname}: {e}')
        skipped.append(col_name)

print(f'\nExtracted: {len(extracted)} features — {extracted}')
if skipped:
    print(f'Skipped:   {len(skipped)} features — {skipped}')
print(f'Dataframe shape: {df.shape}')

  ✓ elevation: min=nan  max=nan
  ✓ slope: min=nan  max=nan
  ✓ aspect: min=nan  max=nan
  ✓ hillshade: min=nan  max=nan
  ✓ tpi: min=nan  max=nan
  ✓ ndvi: min=nan  max=nan
  ✓ ndwi: min=nan  max=nan
  ✓ evi: min=nan  max=nan
  ✓ nbr: min=nan  max=nan
  ✓ wind_speed: min=nan  max=nan
  ✓ lst: min=nan  max=nan
  ✓ dist_roads: min=nan  max=nan
  ✓ dist_urban: min=nan  max=nan
  ✓ rainfall: min=nan  max=nan
  ✓ soil_moisture: min=nan  max=nan

Extracted: 15 features — ['elevation', 'slope', 'aspect', 'hillshade', 'tpi', 'ndvi', 'ndwi', 'evi', 'nbr', 'wind_speed', 'lst', 'dist_roads', 'dist_urban', 'rainfall', 'soil_moisture']
Dataframe shape: (1045, 16)


## Step 6 — Handle Missing Values

In [33]:
# Replace rasterio nodata sentinels with NaN
df.replace(-9999, np.nan, inplace=True)
df.replace(0, np.nan, inplace=False)  # inspect only — don't drop zeros (valid values)

print('Missing values per column:')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'None')

df_clean = df.dropna()
print(f'\nRows before dropna: {len(df)}  |  after: {len(df_clean)}')

Missing values per column:
elevation         47
slope             47
aspect            47
hillshade         47
tpi               47
ndvi              47
ndwi              47
evi               47
nbr               47
wind_speed        48
lst               47
dist_roads        47
dist_urban       408
rainfall          47
soil_moisture     47
dtype: int64

Rows before dropna: 1045  |  after: 608


## Step 7 — Inspect Dataframe

In [35]:
print('Shape:', df_clean.shape)
print('\nHead:')
print(df_clean.head())
print('\nDescribe:')
print(df_clean.describe().round(3))
print('\nClass balance:')
counts = df_clean['Label'].value_counts()
print(f'  Burned   (1): {counts.get(1, 0)}')
print(f'  Unburned (0): {counts.get(0, 0)}')

Shape: (608, 16)

Head:
    Label  elevation      slope      aspect  hillshade        tpi      ndvi  \
12      1       54.0  14.344421  314.132721      212.0   8.306123  0.738233   
13      1       66.0  15.258314  110.857979      136.0  13.979591  0.627017   
14      1       31.0   5.593202  335.386322      188.0   1.102041  0.460372   
15      1       38.0  14.008526  101.224480      137.0 -14.714286  0.627017   
16      1      118.0  16.841084  282.351685      231.0  39.632652  0.486124   

        ndwi       evi       nbr  wind_speed        lst  dist_roads  \
12 -0.673734  0.497708  0.497155    0.590673  35.772320         0.0   
13 -0.609698  0.394220  0.329588    0.590673  36.661003         0.0   
14 -0.563725  0.275467  0.165031    0.590673  40.624767         0.0   
15 -0.609698  0.394220  0.329588    0.590673  36.661003         0.0   
16 -0.527454  0.368281  0.217507    0.590673  31.709433         0.0   

      dist_urban    rainfall  soil_moisture  
12  10685.062500  619.737732

## Step 8 — Save Inputs.txt and Label.txt

In [28]:
feature_cols = [c for c in df_clean.columns if c != 'label']
X = df_clean[feature_cols].values
y = df_clean['Label'].values

np.savetxt(f'{DRIVE}/Inputs.txt', X)
np.savetxt(f'{DRIVE}/Label.txt',  y)

print('Saved:')
print(f'  {DRIVE}/Inputs.txt  shape={X.shape}')
print(f'  {DRIVE}/Label.txt   shape={y.shape}')
print('\nFeature column order:')
for i, col in enumerate(feature_cols):
    print(f'  col {i:2d}: {col}')
print('\nDone. Next: 02_ml_training.ipynb')

Saved:
  /content/drive/MyDrive/GIS_Wildfire_Mugla/Inputs.txt  shape=(608, 16)
  /content/drive/MyDrive/GIS_Wildfire_Mugla/Label.txt   shape=(608,)

Feature column order:
  col  0: Label
  col  1: elevation
  col  2: slope
  col  3: aspect
  col  4: hillshade
  col  5: tpi
  col  6: ndvi
  col  7: ndwi
  col  8: evi
  col  9: nbr
  col 10: wind_speed
  col 11: lst
  col 12: dist_roads
  col 13: dist_urban
  col 14: rainfall
  col 15: soil_moisture

Done. Next: 02_ml_training.ipynb
